## Decorator Pattern Overview

In [26]:
from abc import ABC, abstractmethod
import time
import random

# Project: Notification system (using decorator pattern)

In [29]:
class Notification:
    @abstractmethod
    def send(self, message):
        pass

class EmailNotification(Notification):
    def send(self, message):
        # Simulate random failure (30% chance)
        if random.random() < 0.3:
            raise Exception("Network error: Failed to send email")
        print(f"Sending Email Notification: {message}")

class BaseDecorator(Notification):
    def __init__(self, notification_method: Notification):
        self.notification_method = notification_method

    @abstractmethod
    def send(self, message):
        return self.notification_method.send(message)

class AuthenticationDecorator(BaseDecorator):
    def send(self, message):
        # Perform authentication logic here
        print("Authenticating user...")
        super().send(message)
    
class RetryDecorator(BaseDecorator):
    def __init__(self, notification_method: Notification, retries: int = 3):
        super().__init__(notification_method)
        self.retries = retries

    def send(self, message):
        for attempt in range(self.retries):
            try:
                super().send(message)
                break  # Exit loop if successful
            except Exception as e:
                print(f"Attempt {attempt + 1} failed: {e}")
                if attempt == self.retries - 1:
                    print("All attempts failed.")

class LoggingDecorator(BaseDecorator):
    def send(self, message):
        print("LOG: Before sending")
        super().send(message)
        print("LOG: After sending")


class MetricsDecorator(BaseDecorator):
    def send(self, message):
        start = time.time()

        super().send(message)

        end = time.time()
        print(f"Execution time: {end-start}")

In [30]:
notification = MetricsDecorator(LoggingDecorator(RetryDecorator(AuthenticationDecorator(EmailNotification()), retries=3)))
notification.send("Hi") 


LOG: Before sending
Authenticating user...
Attempt 1 failed: Network error: Failed to send email
Authenticating user...
Sending Email Notification: Hi
LOG: After sending
Execution time: 0.0
